In [1]:
from pathlib import Path
import json
from urllib.request import urlopen
import pandas as pd

project = Path.home() / "Documents" / "BSE_Climate_Policy"

countries = "AUS;CAN;DEU;ESP;FRA;GBR;IRL;ITA;MEX;NZL;USA"
indicator = "EG.FEC.RNEW.ZS"
url = (
    f"https://api.worldbank.org/v2/country/{countries}"
    f"/indicator/{indicator}?format=json&per_page=1000&date=2000:2019"
)

with urlopen(url, timeout=30) as response:
    data = json.load(response)

renew = pd.DataFrame(
    {
        "country_code": row["countryiso3code"],
        "year": int(row["date"]),
        "renewable_share": row["value"],
    }
    for row in data[1]
).sort_values(["country_code", "year"]).reset_index(drop=True)

print("행·열:", renew.shape)
display(renew.head(10))

행·열: (220, 3)


,country_code,year,renewable_share
0,AUS,2000,8.4
1,AUS,2001,8.4
2,AUS,2002,8.7
3,AUS,2003,7.2
4,AUS,2004,6.7
5,AUS,2005,6.7
6,AUS,2006,6.8
7,AUS,2007,7.0
8,AUS,2008,6.8
9,AUS,2009,7.1


In [2]:
print("국가별 결측치:")
print(renew.groupby("country_code")["renewable_share"].apply(lambda x: x.isna().sum()))

output_file = project / "processed_data" / "wdi_renewshare_2000_2019.csv"
renew.to_csv(output_file, index=False)

print("저장 완료:", output_file)

국가별 결측치:
country_code
AUS    0
CAN    0
DEU    0
ESP    0
FRA    0
GBR    0
IRL    0
ITA    0
MEX    0
NZL    0
USA    0
Name: renewable_share, dtype: int64
저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/wdi_renewshare_2000_2019.csv


In [3]:
panel_file = project / "processed_data" / "cpu_carbon_panel_draft_2000_2019.csv"

panel = pd.read_csv(panel_file)

panel_with_renew = panel.merge(
    renew,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

print("합친 데이터 크기:", panel_with_renew.shape)
print("재생에너지 비중 결측치:", panel_with_renew["renewable_share"].isna().sum())
display(panel_with_renew.head())

합친 데이터 크기: (220, 5)
재생에너지 비중 결측치: 0


,country_code,year,CPU,CarbonPrice_draft,renewable_share
0,AUS,2000,37.986378,0.0,8.4
1,AUS,2001,32.302196,0.0,8.4
2,AUS,2002,23.147430,0.0,8.7
3,AUS,2003,32.739329,0.0,7.2
4,AUS,2004,32.399758,0.0,6.7


In [4]:
assert len(panel_with_renew) == 220, "행 수가 220이 아니에요."
assert panel_with_renew["renewable_share"].notna().all(), "재생에너지 값에 빈칸이 있어요."

output_file = project / "processed_data" / "cpu_carbon_renew_panel_draft_2000_2019.csv"
panel_with_renew.to_csv(output_file, index=False)

print("저장 완료:", output_file)
print("데이터 크기:", panel_with_renew.shape)

저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/cpu_carbon_renew_panel_draft_2000_2019.csv
데이터 크기: (220, 5)


In [6]:
gdp_indicator = "NY.GDP.PCAP.KD"
gdp_url = (
    f"https://api.worldbank.org/v2/country/{countries}"
    f"/indicator/{gdp_indicator}?format=json&per_page=1000&date=2000:2019"
)

with urlopen(gdp_url, timeout=90) as response:
    gdp_data = json.load(response)

gdp = pd.DataFrame(
    {
        "country_code": row["countryiso3code"],
        "year": int(row["date"]),
        "gdp_per_capita": row["value"],
    }
    for row in gdp_data[1]
).sort_values(["country_code", "year"]).reset_index(drop=True)

print("행·열:", gdp.shape)
print("결측치:", gdp["gdp_per_capita"].isna().sum())
display(gdp.head())

행·열: (220, 3)
결측치: 0


,country_code,year,gdp_per_capita
0,AUS,2000,45979.076565
1,AUS,2001,46314.623530
2,AUS,2002,47599.407929
3,AUS,2003,48509.663218
4,AUS,2004,50018.590448


In [7]:
gdp_file = project / "processed_data" / "wdi_gdp_per_capita_2000_2019.csv"
gdp.to_csv(gdp_file, index=False)

panel_gdp = panel_with_renew.merge(
    gdp,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

assert len(panel_gdp) == 220
assert panel_gdp["gdp_per_capita"].notna().all()

output_file = project / "processed_data" / "cpu_carbon_renew_gdp_panel_draft_2000_2019.csv"
panel_gdp.to_csv(output_file, index=False)

print("저장 완료:", output_file)
display(panel_gdp.head())

저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/cpu_carbon_renew_gdp_panel_draft_2000_2019.csv


,country_code,year,CPU,CarbonPrice_draft,renewable_share,gdp_per_capita
0,AUS,2000,37.986378,0.0,8.4,45979.076565
1,AUS,2001,32.302196,0.0,8.4,46314.623530
2,AUS,2002,23.147430,0.0,8.7,47599.407929
3,AUS,2003,32.739329,0.0,7.2,48509.663218
4,AUS,2004,32.399758,0.0,6.7,50018.590448


In [8]:
trade_indicator = "NE.TRD.GNFS.ZS"
trade_url = (
    f"https://api.worldbank.org/v2/country/{countries}"
    f"/indicator/{trade_indicator}?format=json&per_page=1000&date=2000:2019"
)

with urlopen(trade_url, timeout=90) as response:
    trade_data = json.load(response)

trade = pd.DataFrame(
    {
        "country_code": row["countryiso3code"],
        "year": int(row["date"]),
        "trade_pct_gdp": row["value"],
    }
    for row in trade_data[1]
).sort_values(["country_code", "year"]).reset_index(drop=True)

print("행·열:", trade.shape)
print("결측치:", trade["trade_pct_gdp"].isna().sum())
display(trade.head())

행·열: (220, 3)
결측치: 0


,country_code,year,trade_pct_gdp
0,AUS,2000,40.863126
1,AUS,2001,44.133731
2,AUS,2002,41.381313
3,AUS,2003,40.134801
4,AUS,2004,36.950516


In [9]:
trade_file = project / "processed_data" / "wdi_trade_pct_gdp_2000_2019.csv"
trade.to_csv(trade_file, index=False)

panel_trade = panel_gdp.merge(
    trade,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

assert len(panel_trade) == 220
assert panel_trade["trade_pct_gdp"].notna().all()

output_file = project / "processed_data" / "cpu_carbon_renew_gdp_trade_panel_draft_2000_2019.csv"
panel_trade.to_csv(output_file, index=False)

print("저장 완료:", output_file)
display(panel_trade.head())

저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/cpu_carbon_renew_gdp_trade_panel_draft_2000_2019.csv


,country_code,year,CPU,CarbonPrice_draft,renewable_share,gdp_per_capita,trade_pct_gdp
0,AUS,2000,37.986378,0.0,8.4,45979.076565,40.863126
1,AUS,2001,32.302196,0.0,8.4,46314.623530,44.133731
2,AUS,2002,23.147430,0.0,8.7,47599.407929,41.381313
3,AUS,2003,32.739329,0.0,7.2,48509.663218,40.134801
4,AUS,2004,32.399758,0.0,6.7,50018.590448,36.950516


In [10]:
industry_indicator = "NV.IND.TOTL.ZS"
industry_url = (
    f"https://api.worldbank.org/v2/country/{countries}"
    f"/indicator/{industry_indicator}?format=json&per_page=1000&date=2000:2019"
)

with urlopen(industry_url, timeout=90) as response:
    industry_data = json.load(response)

industry = pd.DataFrame(
    {
        "country_code": row["countryiso3code"],
        "year": int(row["date"]),
        "industry_pct_gdp": row["value"],
    }
    for row in industry_data[1]
).sort_values(["country_code", "year"]).reset_index(drop=True)

print("행·열:", industry.shape)
print("결측치:", industry["industry_pct_gdp"].isna().sum())
display(industry.head())

행·열: (220, 3)
결측치: 0


,country_code,year,industry_pct_gdp
0,AUS,2000,24.532457
1,AUS,2001,23.586695
2,AUS,2002,23.555171
3,AUS,2003,23.991726
4,AUS,2004,23.881909


In [11]:
industry_file = project / "processed_data" / "wdi_industry_pct_gdp_2000_2019.csv"
industry.to_csv(industry_file, index=False)

panel_industry = panel_trade.merge(
    industry,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

assert len(panel_industry) == 220
assert panel_industry["industry_pct_gdp"].notna().all()

output_file = project / "processed_data" / "cpu_carbon_renew_gdp_trade_industry_panel_draft_2000_2019.csv"
panel_industry.to_csv(output_file, index=False)

print("저장 완료:", output_file)
display(panel_industry.head())

저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/cpu_carbon_renew_gdp_trade_industry_panel_draft_2000_2019.csv


,country_code,year,CPU,CarbonPrice_draft,renewable_share,gdp_per_capita,trade_pct_gdp,industry_pct_gdp
0,AUS,2000,37.986378,0.0,8.4,45979.076565,40.863126,24.532457
1,AUS,2001,32.302196,0.0,8.4,46314.623530,44.133731,23.586695
2,AUS,2002,23.147430,0.0,8.7,47599.407929,41.381313,23.555171
3,AUS,2003,32.739329,0.0,7.2,48509.663218,40.134801,23.991726
4,AUS,2004,32.399758,0.0,6.7,50018.590448,36.950516,23.881909


In [12]:
carbon_indicator = "EN.GHG.CO2.RT.GDP.PP.KD"
carbon_url = (
    f"https://api.worldbank.org/v2/country/{countries}"
    f"/indicator/{carbon_indicator}?format=json&per_page=1000&date=2000:2019"
)

with urlopen(carbon_url, timeout=90) as response:
    carbon_data = json.load(response)

carbon_intensity = pd.DataFrame(
    {
        "country_code": row["countryiso3code"],
        "year": int(row["date"]),
        "carbon_intensity": row["value"],
    }
    for row in carbon_data[1]
).sort_values(["country_code", "year"]).reset_index(drop=True)

print("행·열:", carbon_intensity.shape)
print("결측치:", carbon_intensity["carbon_intensity"].isna().sum())
display(carbon_intensity.head())

행·열: (220, 3)
결측치: 0


,country_code,year,carbon_intensity
0,AUS,2000,0.412379
1,AUS,2001,0.411628
2,AUS,2002,0.405011
3,AUS,2003,0.392333
4,AUS,2004,0.390602


In [13]:
carbon_file = project / "processed_data" / "wdi_carbon_intensity_2000_2019.csv"
carbon_intensity.to_csv(carbon_file, index=False)

panel_carbon = panel_industry.merge(
    carbon_intensity,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

assert len(panel_carbon) == 220
assert panel_carbon["carbon_intensity"].notna().all()

output_file = project / "processed_data" / "cpu_carbon_renew_gdp_trade_industry_intensity_panel_draft_2000_2019.csv"
panel_carbon.to_csv(output_file, index=False)

print("저장 완료:", output_file)
display(panel_carbon.head())

저장 완료: /Users/imsnmn/Documents/BSE_Climate_Policy/processed_data/cpu_carbon_renew_gdp_trade_industry_intensity_panel_draft_2000_2019.csv


,country_code,year,CPU,CarbonPrice_draft,renewable_share,gdp_per_capita,trade_pct_gdp,industry_pct_gdp,carbon_intensity
0,AUS,2000,37.986378,0.0,8.4,45979.076565,40.863126,24.532457,0.412379
1,AUS,2001,32.302196,0.0,8.4,46314.623530,44.133731,23.586695,0.411628
2,AUS,2002,23.147430,0.0,8.7,47599.407929,41.381313,23.555171,0.405011
3,AUS,2003,32.739329,0.0,7.2,48509.663218,40.134801,23.991726,0.392333
4,AUS,2004,32.399758,0.0,6.7,50018.590448,36.950516,23.881909,0.390602


In [16]:
print("데이터 크기:", panel_carbon.shape)
print("국가 수:", panel_carbon["country_code"].nunique())
print("연도 범위:", panel_carbon["year"].min(), "–", panel_carbon["year"].max())
print("국가·연도 중복:", panel_carbon.duplicated(["country_code", "year"]).sum())

print("\n변수별 결측치:")
print(panel_carbon.isna().sum())

데이터 크기: (220, 9)
국가 수: 11
연도 범위: 2000 – 2019
국가·연도 중복: 0

변수별 결측치:
country_code          0
year                  0
CPU                  13
CarbonPrice_draft     1
renewable_share       0
gdp_per_capita        0
trade_pct_gdp         0
industry_pct_gdp      0
carbon_intensity      0
dtype: int64
